# Setup

In [ ]:
library(data.table)

if (!require('picsure', quiet=T)) {
  Sys.setenv(TAR = if(Sys.info()['sysname']=='Windows') 'C:/WINDOWS/system32/tar' else '/bin/tar')
  options(unzip = 'internal')
  remotes::install_github(
    repo = 'hms-dbmi/pic-sure-r-adapter-hpds',
    ref  = 'main', force=T
  )
  library(picsure)
}

# MANUAL STEP:
You must go to the PIC-SURE website, get an API token, and temporarily copy it into a file named `picsure_token.txt` in the workspace bucket.

In [ ]:
system('gcloud storage cp $WORKSPACE_BUCKET/picsure_token.txt .')

In [ ]:
picsure_url = 'https://picsure.biodatacatalyst.nhlbi.nih.gov/picsure'
token <- scan('picsure_token.txt', what='character', quiet=T)
session <- picsure::bdc.initializeSession(picsure_url, token) |> picsure::bdc.setResource('AUTH')

# Search variables
## Get search objects
This is analogous to exploring variables in the PIC-SURE web UI.

In [ ]:
if(!file.exists('picsure_searches.rds')) {
  # For some reason, query wait time scales with `limit`, so set it as low as possible while still getting all variables.
  t0 <- Sys.time()
   mesa_ids_search <- picsure::bdc.searchPicsure(session, 'phs001416',  limit=  200) |> setDT()
       mesa_search <- picsure::bdc.searchPicsure(session, 'phs000209',  limit=25000) |> setDT()
       fhs1_search <- picsure::bdc.searchPicsure(session, 'phs000007',  limit=60000) |> setDT()
       fhs2_search <- picsure::bdc.searchPicsure(session, 'phs000974',  limit= 2000) |> setDT()
     topmed_search <- picsure::bdc.searchPicsure(session, 'DCC Harmon', limit=  500) |> setDT()
  Sys.time() - t0
  save(mesa_ids_search, mesa_search, fhs1_search, fhs2_search, topmed_search, file='picsure_searches.rds')
} else { load('picsure_searches.rds') }

## Narrow down desired variables
### TOPMed harmonized variables
We don't actually use these, we prefer the MESA- and FHS-specific ones. But, I leave this code here in case it is useful.

In [ ]:
tmp <- topmed_search[,-'name'][, var_description := gsub('\n',' ',var_description)] # For manual exploration if desired.

topmed_vars <- topmed_search[
  ][grepl(
      x   = tolower(paste(var_id, var_name, var_description)),
      pat = paste(collapse='|', c(
        'lipid_lowering_medication',
        'fasting_lipids', #(This is an indicator var of whether they fasted >=8hr b4 lipid measures or not)
        'hdl',
        'triglycerides',
        'bmi_baseline', # 'Body mass index calculated at baseline.'
        'ever_smoker_baseline',
        'current_smoker_baseline',
        'sleep_duration',
        'race_us',
        'subcohort', # 'A distinct subgroup within a study, generally indicating subjects who share similar characteristics due to study design. Subjects may belong to only one subcohort.'
        'hispanic_or_latino',
        'geographic_site',
        'annotated_sex',
        'age_at_hdl', 'age_at_triglycerides', 'age_at_ever_smoker', 'age_at_current_smoker', 'age_at_fasting_lipids', 'age_at_lipid_lowering', 'age_at_sleep'
    )))
]

### FHS (phs000007) search

In [ ]:
tmp <- fhs1_search[,-'name'][study_id=='phs000007'] # For manual exploration.

fhs1_vars <- fhs1_search[
  ][study_id=='phs000007'
    # We only care about Offspring exam 5 & Gen3 exam 1,
    #   the only ones with metabolomics timepoints.
  ][grepl('pht006026|pht009761',name)
  ][grepl(
      x   = tolower(paste(var_id, var_name, var_description)),
      pat = paste(collapse='|', c(
        'shareid',
        'sex',
        'age',
        'bmi',
        'smoking',
        'mvpa',
        'glucose',
        'a1c',
        'insulin',
        'hdl',
        'triglyceride',
        'lipid_med',
        'race'
    )))
]

### FHS (phs000974) search

In [ ]:
tmp <- fhs2_search[,-'name'][, var_description := gsub('\n',' ',var_description)] # For manual exploration.

fhs2_vars <- fhs2_search[
  ][grepl(
      x   = var_name,
      pat = paste(collapse='|', c(
        'Age_at_collection', # There's multiple
        'AGE_OF_COLLECTION',
        'Batch',
        'Collection_visit', # There's multiple
        'extractDate',
        'IS_TUMOR', # all are "no"
        'SEX',
        'SAMPLE_ID',
        'PlateId', 'Sample_well_ID', 'Sample_container_ID',
        'SUBJECT_ID'
    )))
]

### MESA search

In [ ]:
tmp <- mesa_search[,-'name'][, var_description := gsub('\n',' ',var_description)] # For manual exploration.

mesa_vars <- mesa_search[
  ][grepl(
      x   = group_name,
      pat = paste(collapse='|', c(
        'MESA_Exam[0-9]Main',
        'MESA_AncilMesaMineralMetabolite',        # nac
        'MESA_AncilMesaNeighborScalesExam',       # ahei_2010
        'MESA_AncilMesaNeighborScalesSodiumExam', # dash_sodium
        'MESA_exam[0-9]dietnutrients',            # nan
        'MESA_AncilMesaNeighborSESExam'           # F1_PC2
    )))
  ][grepl(
      x   = var_name,
      pat = paste(collapse='|', c(
        'sidno',
        '^age[0-9]c$',
        'ahei',
        'alc',
        'bmi[0-9]c',
        'cig[0-9]c',
        'dash_sodium',
        'F1_PC2',
        'gender',
        'glucos[0-9]c',
        'hba',
        'hdl[0-9]$',
        'income',
        'insln[0-9]c',
        'lipid',
        'month',
        'nac',
        'nan[0-9]c',
        'race1c',
        'season',
        'site[0-9]c',
        'trig[0-9]',
        'exercm[0-9]c', 'pamcm[0-9]c', 'pavcm[0-9]c', 'pamvcm[0-9]c'
    )))
  #][, unique(group_name)
  #][grep('sidno', var_name)
]

In [ ]:
tmp <- mesa_ids_search[,-'name']

mesa_id_vars <- mesa_ids_search[
  ][grepl(
      x   = group_name,
      pat = paste(collapse='|', c(
      'TOPMed_MESA_Sample',
      'TOPMed_MESA_Subject',
      'TOPMed_MESA_Metabolomics'
    )))
]

# Query
Now we actually request the individual-level data for the desired variables with a query:

In [ ]:
do_query <- \(varnames) bdc.newQuery(session) |> addClause(keys=varnames, type='ANYOF') |> runQuery(resultType='DATA_FRAME') |> setDT()

if(!file.exists('picsure_queries.rds')) {
  t0 <- Sys.time()
  mesa_id_query <- do_query(mesa_id_vars$name)
     mesa_query <- do_query(   mesa_vars$name)
     fhs1_query <- do_query(   fhs1_vars$name)
     fhs2_query <- do_query(   fhs2_vars$name)
   topmed_query <- do_query( topmed_vars$name)
  Sys.time() - t0
  save(mesa_id_query,mesa_query,fhs1_query,fhs2_query,topmed_query,file='picsure_queries.rds')
} else { load('picsure_queries.rds') }

# Format phs000007 query results
We only care about Offspring exam 5 & Gen3 exam 1, because they are the only ones with metabolomics timepoints.

There is no site/center/location variable, so we will skip that. \
There are many "drinks per week"-ish variables, but they don't exactly match ("drinks per week" vs. "times drank per week"), so I will omit this covariate to save time harmonizing.

In [ ]:
fhs1_query[fhs1_query==''] <- NA # Otherwise '' will mess with fcoalesce
fhs1_query <- fhs1_query[
  ][, age := fcoalesce(
        `\\phs000007\\pht006026\\phv00277020\\AGE1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544734\\age5\\`) # Offspring
  ][, bmi := fcoalesce(
        `\\phs000007\\pht006026\\phv00277024\\BMI1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544754\\bmi5\\`) # Offspring
  ][, SEX := fcoalesce(
        `\\phs000007\\pht006026\\phv00277017\\SEX\\`, # Gen3
        `\\phs000007\\pht009761\\phv00423870\\SEX\\`) # Offspring
  ][, hdl := fcoalesce(
        `\\phs000007\\pht006026\\phv00277040\\HDL1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544810\\HDL5\\`) # Offspring
  ][, tg  := fcoalesce(
        `\\phs000007\\pht006026\\phv00277049\\TRIG1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544856\\TRIG5\\`) # Offspring
  ][, fg  := fcoalesce(
        `\\phs000007\\pht006026\\phv00277038\\FASTING_BG1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544804\\FASTING_BG5\\`) # Offspring
  ][, smoking := fcoalesce(
        `\\phs000007\\pht006026\\phv00277032\\CURRSMK1\\`, # Gen3
        `\\phs000007\\pht009761\\phv00544778\\currsmk5\\`) # Offspring
  ][, .SD, .SDcols = patterns(paste(collapse='|', c(
      '^age$', '^bmi$', '^SEX$', '^hdl$', '^tg$', '^fg$', '^smoking$',
      'patient_id', 'Study Accession')))
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm))

# Deal with duplicate columns in FHS (phs000974)
FHS has many columns with duplicate names and their distinction is not clear from the PIC-SURE `variable_description`s. We must manually [look up their phv #s in dbGap](https://dbgap.ncbi.nlm.nih.gov/beta/search/) or in some cases infer by exploring the data ourselves. \
Strategy: I subset to sets of same-name columns, manually explored the data in VisiData, and then added R snippets after the fact to demonstrate any interesting properties/differences I discovered.

### `SUBJECT_ID`
There are 3 `SUBJECT_ID` cols. A dbGaP search of their phv #s reveals no useful info to distinguish them. \
The 3 columns are identical, except the 2nd one has 8 NAs in it, and the 3rd one has 6407 NAs in it. \
In short, the 1st column has the most complete information and is in fact a superset of the other two, so just use that one (phv00252386).

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('SUBJECT_ID')
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) # For manual viewing

tmp[, sapply(.SD,\(x)sum(is.na(x)))] # Count NAs

# Showing the non-NA ids in cols 2&3 are identical to the complete col 1.
tmp[complete.cases(tmp[,c(1,2)]), identical(.SD[[1]],.SD[[2]])] # TRUE
tmp[complete.cases(tmp[,c(1,3)]), identical(.SD[[1]],.SD[[3]])] # FALSE!?
tmp[complete.cases(tmp[,c(1,3)]), identical(.SD[[1]],as.integer(.SD[[3]]))] # Oh, just integer vs. numeric

### `IS_TUMOR`
There are 4 `IS_TUMOR` columns. \
They all have missing data in diffierent places. Probably refer to different omics. \
But nothing is a tumor sample anyway, so forget about it.

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('IS_TUMOR')
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) # For manual viewing

tmp[, sapply(.SD,table)] # Always "no" or NA.

### `Collection_visit`
There are 4 Collection_visit columns. \
A dbGaP search reveals [phv00543910](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543910) and [phv00543919](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543919) are the ones relevant to metabolomics which we want. \
Why are there two metabolomics ones? Dunno. But the two columns do not have any conflicts, we can can coalesce them.

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('Collection_visit')
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) # For manual viewing

# All 4 cols are almost perfectly coalesce-able, except 12 people with a mismatching visit "1" and "2".
# The phv00543910 and phv00543919 are the metabolomics ones. Thankfully no mismatches b/t those two.
tmp[sample(1:.N,5)] # 5 random rows
tmp[, .SD[ .SD[[1]]!=.SD[[4]] ]] # mismatches
tmp[, .SD[ .SD[[3]]!=.SD[[4]] ]] # But no mismatches b/t the two metabolomics ones, phew.

### `Sample_container_ID` & `Sample_well_ID`
There are 2 of each. \
These containers must refer to different omics. If you search up the dbGaP phv #s, you can see. \
The 1st col ("RAMA___ containers") is [phv00500015](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00500015), methylomics. The 2nd col (plain-numbered) is [phv00543928](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543928), metabolomics.

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('Sample_container_ID|Sample_well_ID')
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) # For manual viewing

tmp[sample(1:.N,9)] # 9 random rows

#### `PlateId` & `Batch` & `extractDate`
There's also `PlateId` ([phv00498670](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00498670)) and `Batch` ([phv00543896](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543896)) & `extractDate` ([phv00543899](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543899)), but those are for proteomics/methylomics/methylomics.

### `Age_at_collection` & `AGE_OF_COLLECTION`
There are 4 of these columns total. \

* [phv00500009](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00500015) -- Methylomics
* [phv00543906](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543906) -- RNASeq
* [phv00543908](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543908) -- Metabolomics
* [phv00543917](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543917) -- Metabolomics also

Below simply confirms that the ages of the two metabolomics columns have no conflicts.

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('AGE_OF_COLLECTION|Age_at_collection')
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm))
# (Manually confirmed there are no conflicts between the two metabolomics columns externally in VisiData and I'm lazy to add demonstrative code here.)

### `SAMPLE_ID`
There are 6 `SAMPLE_ID` columns. \
The first column is simply all the other columns concatenated with tabs. The others are NWD (phv00252391), TOE (phv00499999), TOR (phv00500017), TOM(phv00543907), and TOM(phv00543916) (--yes, there are two TOM ones). \
The two TOM ones have no conflicts, so I will just coalesce them, I guess. \

There are also 9 people with 2 or more tab-separated NWD ids. It is not clear which is the preferred one, so I will arbitrarily take the first. It is not enough samples to warrant the merging complications that would arise from trying to consider them all.

In [ ]:
tmp <- fhs2_query[, .SD, .SDcols=patterns('SAMPLE_ID')
][, names(.SD) := lapply(.SD, \(x) gsub('\t',' ',x))
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) # For manual viewing

tmp[, .SD[[2]][grepl(' ',.SD[[2]])]] # Those with multiple NWD ids

tmp[sample(1:.N,9)] # 9 random rows

## Putting all of this together to tidy the FHS variables
In conclusion since we are only interested in metabolomics this is what we'll do: \

* `SEX`: Keep
* `SUBJECT_ID`: Keep only the phv00252386 one
* `IS_TUMOR` & `PlateId` & `Batch`: Drop
* `Collection_visit`: Coalesce [phv00543910](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543910) & [phv00543919](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543919)
* `Age_at_collection`: Coalesce [phv00543908](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543908) & [phv00543917](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543917)
* `SAMPLE_ID`: Coalesce [phv00543907](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543907) & [phv00543916](https://dbgap.ncbi.nlm.nih.gov/beta/search/?OBJ=variable&TERM=phv00543916)

In [ ]:
fhs2_query[fhs2_query==''] <- NA # Otherwise '' will mess with fcoalesce
fhs2_query <- fhs2_query[
  ][, metabolomics_age := fcoalesce(
        `\\phs000974\\pht015035\\phv00543908\\Age_at_collection\\`, 
        `\\phs000974\\pht015036\\phv00543917\\Age_at_collection\\`)
  ][, metabolomics_visit := fcoalesce(
        `\\phs000974\\pht015035\\phv00543910\\Collection_visit\\`,
        `\\phs000974\\pht015036\\phv00543919\\Collection_visit\\`)
  ][, TOM_Id := fcoalesce(
        `\\phs000974\\pht015035\\phv00543907\\SAMPLE_ID\\`,
        `\\phs000974\\pht015036\\phv00543916\\SAMPLE_ID\\`)
  ][, NWD_Id := sub('\t.*','', `\\phs000974\\pht004911\\phv00252391\\SAMPLE_ID\\`) # Keep only the 1st if multiple tab-delimited NWD ids.
  ][, .SD, .SDcols = patterns(paste(collapse='|', c(
    'SEX', 'patient_id', 'Study Accession',
    'phv00252386', # SUBJECT_ID
    'metabolomics_age', 'metabolomics_visit', 'NWD_Id', 'TOM_Id')))
] |> setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm))

# Merge the two FHS datasets together

In [ ]:
fhs_full <- fhs1_query[
  ][fhs2_query,
    on=.(`_Topmed Study Accession with Subject ID`,
         `_Parent Study Accession with Subject ID`,
         patient_id, SEX)
]

# Write

In [ ]:
dir.create('data/derived/phenotypes', rec=T)
fwrite(mesa_query,      'data/derived/phenotypes/mesa_picsure_variables.csv')
fwrite(mesa_id_query,   'data/derived/phenotypes/mesa_id_picsure_variables.csv')
fwrite(fhs_full,        'data/derived/phenotypes/fhs_picsure_variables.csv')
fwrite(topmed_query,    'data/derived/phenotypes/topmed_picsure_variables.csv')
system('gcloud storage cp -r data/derived/phenotypes $WORKSPACE_BUCKET')